# sLLM

**소형 언어 모델(sLLM)** 은 매개변수 수를 수십억에서 수백억 대로 제한하여, 대형 언어 모델(LLM)이 제공하는 핵심 기능을 유지하면서도 경량화·고효율화를 추구한 언어 모델이다.  

- **경량화**: 수십억~수백억 규모의 파라미터로 구성되어, LLM 대비 메모리·연산 요구량이 대폭 감소한다.  
- **비용 효율성**: 학습·추론 비용이 낮고, 전력 소모가 적어 온프레미스나 엣지 환경에서 활용하기 적합하다.  
- **실시간성**: 파라미터 수가 적어 추론 속도가 빠르며, 모바일·노트북·임베디스 환경에서도 운영 가능하다.  
- **도메인 특화**: 특정 분야 데이터로 미세조정(fine-tuning)하면 LLM과 유사한 성능을 달성할 수 있다.  

**1. 정의 및 분류**

소형 언어 모델은 파라미터 수가 수십억~수백억 대이며, 일반적으로 다음 세 가지 범주로 구분된다:  
- LLM(매개변수 ≥1천억): 최고 성능·높은 자원 요구  
- sLLM(수십~수백억): 성능·자원 효율 균형  
- sLM(수십억 이하): 단순 작업·극한 제약 환경 특화  

|구분   |파라미터 수       |주요 특징                         |
|-------|------------------|-----------------------------------|
|LLM    |≥1,000억          |최고 성능·고자원                  |
|sLLM   |수십~수백억       |경량화·비용 효율성·응답 속도 우수 |
|sLM    |수십억 이하       |극단적 경량화·제한적 기능         |

**2. 주요 특징**

1. **경량화**  
   - 파라미터 수 감소로 모델 크기·메모리 점유율 최소화  
   - 예: TinyLlama는 11억 파라미터, 4비트 양자화 시 550 MB RAM 차지.  

2. **비용·자원 효율성**  
   - GPU·클라우드 비용 절감  
   - 온프레미스·모바일 환경에서도 실시간 추론 가능.  

3. **도메인 특화 및 미세조정**  
   - 한국어·금융·의료 등 특정 분야 데이터로 추가 학습해 고성능 발휘  
   - 예: beomi/Yi-Ko-6B(6 B 매개변수) 모델은 한국어·영어 병합 데이터로 학습되어 한국어 작업에서 우수한 성능 보유.  

4. **응답 속도**  
   - 매개변수 수가 적어 LLM 대비 추론 속도 2~10배 향상  

**3. 대표적인 sLLM 사례**

|모델명           |파라미터 수 |특징                                                         |
|----------------|----------|-------------------------------------------------------------|
|TinyLlama       |1.1 B     |Llama2 아키텍처 기반, FlashAttention 적용, 4비트 양자화 가능.   |
|Yi-Ko-6B        |6 B       |한국어·영어 혼합 사전학습, 4 K 컨텍스트 길이, HuggingFace 지원.  |
|Mistral 7B      |7.3 B     |영어·코딩 작업 특화, MMLU 60.1% 기록, Apache 2.0 라이선스 공개.    |
|Phi-2           |2.7 B     |MS 온디바이스 AI용, 추론 최적화, 완전 오픈소스.             |
|Motif 2.6B      |2.6 B     |국산 모델, Mistral 7B 대비 134% 우수 성능(지디넷).            |

**4. 과제 및 한계**

- **추론 능력**: 창의적·복합 문제 해결에서 LLM 대비 성능 격차 존재  
- **언어별 편향**: 한국어·소저자원 언어에서 데이터 부족 시 성능 저하 가능  
- **모델 안전성**: 판별 어려운 환각 위험 관리 필요  

## 패키지 설치

In [1]:
!pip install transformers huggingface_hub accelerate datasets openai

  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.15.0-py3-none-any.whl.metadata (14 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached openai-2.37.0-py3-none-any.whl.metadata (31 kB)
  Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using ca

## 환경 변수

In [2]:
import os

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
HF_TOKEN = os.environ['HF_TOKEN']

## Llama Model 테스트

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = 'meta-llama/Llama-3.2-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.bfloat16,
        device_map='auto', # 사용가능한 gpu 자동할당
)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [6]:
model.device

device(type='cuda', index=0)

In [7]:
prompt = 'Explain the difference between high and low tides.'

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
print(inputs)

{'input_ids': tensor([[128000,    849,  21435,    279,   6811,   1990,   1579,    323,   3428,
            259,   3422,     13]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


In [8]:
outputs = model.generate(**inputs, max_length=1024)
print(outputs)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tensor([[128000,    849,  21435,    279,   6811,   1990,   1579,    323,   3428,
            259,   3422,     13,    578,   1925,   9547,    430,   7958,    279,
            259,   3422,    527,    279,  71019,   6958,    315,    279,  18266,
            323,   7160,     11,    323,    279,   6211,    315,    279,  80944,
            627,    334,  16533,     25,   1035,    334,  12243,  43038,  68063,
            362,   1579,  43038,  13980,    994,    279,  71019,   6958,    315,
            279,  18266,    323,   7160,  11384,    279,  18435,   3090,    311,
           7173,    713,    704,   7119,    279,  31284,     13,   1115,  11705,
            264,  10205,    304,    279,   9581,   2237,     11,  13239,    304,
            264,   5190,   3090,   2237,     13,    578,   6211,    315,    279,
          80944,    323,    279,   8149,    315,    279,  18435,   1101,   1514,
            264,   3560,    304,  26679,    279,   1579,  43038,    627,    334,
          25162,  43038,  68

In [9]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


"Explain the difference between high and low tides. The main factors that affect the tides are the gravitational pull of the moon and sun, and the shape of the coastline.\n**Answer:**\n**High tide:** A high tide occurs when the gravitational pull of the moon and sun causes the ocean water to bulge out towards the shore. This creates a rise in the sea level, resulting in a higher water level. The shape of the coastline and the depth of the ocean also play a role in determining the high tide.\n**Low tide:** A low tide occurs when the gravitational pull of the moon and sun causes the ocean water to recede away from the shore. This creates a fall in the sea level, resulting in a lower water level. The shape of the coastline and the depth of the ocean also play a role in determining the low tide.\n**Factors affecting tides:**\n* Gravitational pull of the moon and sun\n* Shape of the coastline\n* Depth of the ocean\n* Wind and atmospheric pressure\n* Earth's rotation\n**Note:** The combinati